<style>
.markdown-content {
    direction: rtl;
    text-align: right;
}

.markdown-content pre {
    direction: ltr;
    text-align: left;
}

.markdown-content code {
    direction: ltr;
    text-align: left;
}
</style>

<div class="markdown-content">

نکات اجرای کد:
- ابتدا باید در LM Studio یک مدل Chat را Load کرده و Local Server را فعال کنی.
- مقدار MODEL_ID را با شناسه واقعی مدل بارگذاری‌شده جایگزین کن.
- در این مثال Query به انگلیسی بازنویسی می‌شود تا با اسناد انگلیسی Knowledge Base هماهنگ باشد. اگر اسناد چندزبانه باشند، باید زبان خروجی را متناسب با آن‌ها تنظیم کنی.

In [3]:
import numpy as np
from openai import OpenAI
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# 1. Connect to LM Studio
client = OpenAI(
    base_url="http://localhost:1234/v1",
    api_key="lm-studio"
)

# Replace this with the model identifier shown in LM Studio
MODEL_ID = "google/gemma-4-e4b"

# 2. HR Knowledge Base
documents = [
    {
        "id": "HR-001",
        "text": "Employees receive 26 annual leave days."
    },
    {
        "id": "HR-002",
        "text": "Leave requests must be approved by the manager."
    },
    {
        "id": "HR-003",
        "text": "Employees have supplementary health insurance."
    },
    {
        "id": "HR-004",
        "text": "Monthly salaries are paid at the end of each month."
    },
    {
        "id": "HR-005",
        "text": "Employees can work remotely two days per week."
    },
    {
        "id": "HR-006",
        "text": "Employees can request unpaid leave."
    }
]

# 3. Initialize Embedding Model
embedding_model = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

document_embeddings = embedding_model.encode(
    [doc["text"] for doc in documents],
    convert_to_numpy=True,
    normalize_embeddings=True
)

# 4. Query Rewriting
def rewrite_query(user_query):
    response = client.chat.completions.create(
        model=MODEL_ID,
        temperature=0.0,
        messages=[
            {
                "role": "system",
                "content": """
                You are a query rewriting component for an HR RAG system.

                Rewrite the user's question into a clear, standalone
                search query suitable for retrieving HR documents.

                Rules:
                - Preserve the original intent.
                - Do not answer the question.
                - Do not invent facts or policies.
                - Return only the rewritten query.
                """
            },
            {
                "role": "user",
                "content": user_query
            }
        ]
    )

    return response.choices[0].message.content.strip()


# 5. Semantic Retrieval
def semantic_search(query, top_k=3):
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    scores = cosine_similarity(
        query_embedding,
        document_embeddings
    )[0]

    ranked_indices = np.argsort(scores)[::-1][:top_k]

    return [
        {
            **documents[index],
            "score": float(scores[index])
        }
        for index in ranked_indices
    ]


# 6. Execute Query Rewriting + Retrieval
user_query = "What should I do if I want to take a few days off?"

rewritten_query = rewrite_query(user_query)

results = semantic_search(rewritten_query, top_k=3)

print("Original Query:")
print(user_query)

print("\nRewritten Query:")
print(rewritten_query)

print("\nRetrieved Documents:")

for rank, result in enumerate(results, start=1):
    print(f"\nRank: {rank}")
    print("ID:", result["id"])
    print("Text:", result["text"])
    print("Score:", round(result["score"], 4))

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 780.52it/s]


Original Query:
What should I do if I want to take a few days off?

Rewritten Query:
How do I request time off or vacation leave?

Retrieved Documents:

Rank: 1
ID: HR-006
Text: Employees can request unpaid leave.
Score: 0.5605

Rank: 2
ID: HR-001
Text: Employees receive 26 annual leave days.
Score: 0.4857

Rank: 3
ID: HR-005
Text: Employees can work remotely two days per week.
Score: 0.4303
